# ESL Learning Analysis — ChatGPT Proofreading History

| Field | Value |
|-------|-------|
| **Author** | Haewon |
| **Last update** | 2026-03-22 |
| **Objective** | Extract English learning insights from ChatGPT proofreading history |
| **Sections** | 1. Parse pairs · 2. Pattern analysis · 3. Before/after diff · 4. Anki export |

## 0 — Setup & Load

In [ ]:
import json
import re
import glob
import difflib
import os
import csv
from datetime import datetime
from collections import defaultdict, Counter

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130

CONV_DIR = "/tmp/chatgpt_extract"
OUT_DIR  = "/Users/haewon.yum/Documents/ESL"

PROOFREAD_KEYWORDS = [
    "proofread", "proof read", "grammar", "correct my", "fix my", "check my",
    "rewrite", "rephrase", "improve my", "edit this", "revise", "natural english",
    "sound natural", "native speaker", "more professional", "better way to say",
    "is this correct", "does this sound", "awkward", "more fluent",
    "교정", "영어", "문법", "수정",
]

In [ ]:
def get_messages(conversation):
    mapping = conversation.get("mapping", {})
    current = conversation.get("current_node")
    chain, visited = [], set()
    node_id = current
    while node_id and node_id not in visited:
        visited.add(node_id)
        node = mapping.get(node_id, {})
        msg = node.get("message")
        if msg:
            chain.append(msg)
        node_id = node.get("parent")
    chain.reverse()
    return chain

def get_text(message):
    content = message.get("content", {}) or {}
    parts = content.get("parts", [])
    texts = []
    for p in parts:
        if isinstance(p, str):
            texts.append(p)
        elif isinstance(p, dict):
            texts.append(p.get("text") or p.get("content") or "")
    return "\n".join(texts).strip()

def is_proofread(text):
    tl = text.lower()
    return any(kw in tl for kw in PROOFREAD_KEYWORDS)

def fmt_date(ts):
    try:
        return datetime.fromtimestamp(float(ts)).strftime("%Y-%m-%d")
    except:
        return "unknown"

def load_pairs():
    pairs = []
    files = sorted(glob.glob(f"{CONV_DIR}/conversations-*.json"))
    for fpath in files:
        with open(fpath) as f:
            convs = json.load(f)
        for conv in convs:
            msgs = get_messages(conv)
            for i, msg in enumerate(msgs):
                if msg.get("author", {}).get("role") != "user":
                    continue
                user_text = get_text(msg)
                if not user_text or not is_proofread(user_text):
                    continue
                for j in range(i+1, len(msgs)):
                    nm = msgs[j]
                    if nm.get("author", {}).get("role") == "assistant":
                        asst = get_text(nm)
                        if asst:
                            pairs.append({
                                "conv_id":   conv.get("id", ""),
                                "title":     conv.get("title", ""),
                                "date":      fmt_date(msg.get("create_time")),
                                "user":      user_text,
                                "assistant": asst,
                            })
                        break
    return pairs

pairs = load_pairs()
df = pd.DataFrame(pairs)
print(f"Loaded {len(df)} proofreading pairs")
df.head(3)

## 1 — Extract Corrected Text from Assistant Response

ChatGPT typically puts the corrected version in a fenced block or after a separator. We extract it here.

In [ ]:
def extract_corrected(assistant_text):
    """
    Try to extract just the corrected text from ChatGPT's response.
    Handles common patterns:
      - Text inside --- separators
      - Triple-backtick code blocks
      - Blockquotes (> lines)
      - After intro phrases like 'Here's the corrected version:'
    Returns the best candidate, or the full response if no pattern matches.
    """
    text = assistant_text.strip()

    # Pattern 1: between --- separators (most common ChatGPT format)
    sep_matches = re.findall(r'---+\s*\n(.*?)\n\s*---+', text, re.DOTALL)
    if sep_matches:
        return max(sep_matches, key=len).strip()

    # Pattern 2: triple backtick block
    code_matches = re.findall(r'```(?:\w*)\n(.*?)```', text, re.DOTALL)
    if code_matches:
        return max(code_matches, key=len).strip()

    # Pattern 3: blockquotes
    bq_lines = [l.lstrip('> ') for l in text.split('\n') if l.startswith('>')]
    if len(bq_lines) >= 2:
        return '\n'.join(bq_lines).strip()

    # Pattern 4: after intro phrase
    intro = re.search(
        r'(?:here(?:\'s| is) (?:the )?(?:corrected|revised|improved|proofread|rewritten)[^:\n]*:?|'
        r'corrected version:?|revised version:?)\s*\n+',
        text, re.IGNORECASE
    )
    if intro:
        after = text[intro.end():].strip()
        # Take until next heading-like line or double newline
        chunk = re.split(r'\n{3,}|\n(?:#{1,3} |\*\*[A-Z])', after)[0]
        if len(chunk) > 30:
            return chunk.strip()

    return ""  # couldn't extract cleanly


def extract_user_draft(user_text):
    """
    Strip the proofreading instruction from the user's message to get just the draft.
    e.g. 'proofread:\n\nHi team...' -> 'Hi team...'
    """
    # Remove leading instruction line(s)
    cleaned = re.sub(
        r'^(?:proofread|please proofread|can you proofread|fix|correct|revise|improve|rewrite|check)[^\n]*\n+',
        '', user_text.strip(), flags=re.IGNORECASE
    )
    return cleaned.strip()


df['draft']     = df['user'].apply(extract_user_draft)
df['corrected'] = df['assistant'].apply(extract_corrected)

# How many had clean extractions?
extracted = df[df['corrected'].str.len() > 20]
print(f"Clean corrected text extracted: {len(extracted)} / {len(df)} ({100*len(extracted)//len(df)}%)")
extracted[['date','title','draft','corrected']].head(3)

## 2 — Before / After Diff

Compare draft vs corrected word-by-word and display highlighted diffs.

In [ ]:
from IPython.display import HTML, display

def word_diff_html(original, corrected):
    """Return HTML showing word-level diff (red=removed, green=added)."""
    orig_words = original.split()
    corr_words = corrected.split()
    matcher = difflib.SequenceMatcher(None, orig_words, corr_words)
    html = []
    for op, i1, i2, j1, j2 in matcher.get_opcodes():
        if op == 'equal':
            html.append(' '.join(orig_words[i1:i2]))
        elif op == 'replace':
            html.append(f'<span style="background:#fdd;text-decoration:line-through;color:#900">{" ".join(orig_words[i1:i2])}</span>')
            html.append(f'<span style="background:#dfd;color:#060">{" ".join(corr_words[j1:j2])}</span>')
        elif op == 'delete':
            html.append(f'<span style="background:#fdd;text-decoration:line-through;color:#900">{" ".join(orig_words[i1:i2])}</span>')
        elif op == 'insert':
            html.append(f'<span style="background:#dfd;color:#060">{" ".join(corr_words[j1:j2])}</span>')
    return ' '.join(html)


def show_diffs(n=10, min_len=50):
    """Display n random proofreading diffs that have clean extractions."""
    sample = extracted[extracted['draft'].str.len() >= min_len].sample(n=min(n, len(extracted)), random_state=42)
    for _, row in sample.iterrows():
        diff_html = word_diff_html(row['draft'][:800], row['corrected'][:800])
        display(HTML(f"""
        <div style="border:1px solid #ccc;border-radius:6px;padding:12px;margin:8px 0;font-family:sans-serif">
          <div style="font-size:11px;color:#888;margin-bottom:6px">{row['date']} | {row['title'][:60]}</div>
          <div style="line-height:1.7;font-size:14px">{diff_html}</div>
        </div>
        """))

show_diffs(n=10)

## 3 — Pattern Analysis: What Types of Errors Do You Make?

Classify each word-level change into an error category.

In [ ]:
ARTICLES    = {'a', 'an', 'the'}
PREPOSITIONS = {'in', 'on', 'at', 'for', 'with', 'to', 'of', 'from', 'by', 'about',
                'into', 'through', 'during', 'before', 'after', 'above', 'below',
                'between', 'among', 'under', 'over', 'along', 'around', 'since', 'until'}
CONJUNCTIONS = {'and', 'but', 'or', 'so', 'yet', 'for', 'nor', 'although', 'because',
                'since', 'while', 'whereas', 'however', 'therefore', 'thus'}
PRONOUNS    = {'i', 'me', 'my', 'we', 'us', 'our', 'you', 'your', 'he', 'she', 'it',
               'they', 'them', 'their', 'this', 'that', 'these', 'those'}
BE_VERBS    = {'is', 'are', 'was', 'were', 'be', 'been', 'being', 'am'}

def categorize_change(removed_words, added_words):
    """Assign a category to a single diff chunk."""
    rem = [w.lower().strip('.,;:!?\'"') for w in removed_words]
    add = [w.lower().strip('.,;:!?\'"') for w in added_words]
    rem_set, add_set = set(rem), set(add)

    # Article: added/removed/swapped
    if rem_set <= ARTICLES or add_set <= ARTICLES:
        return 'Article (a/an/the)'
    if rem_set & ARTICLES or add_set & ARTICLES:
        return 'Article (a/an/the)'

    # Preposition
    if (rem_set | add_set) & PREPOSITIONS and len(rem) <= 2 and len(add) <= 2:
        return 'Preposition'

    # Verb tense / form (simple heuristic)
    rem_str, add_str = ' '.join(rem), ' '.join(add)
    if re.search(r'\b(ed|ing|s)\b', rem_str) or re.search(r'\b(ed|ing|s)\b', add_str):
        if len(rem) == 1 and len(add) == 1:
            return 'Verb tense / form'

    # Be verb
    if rem_set <= BE_VERBS or add_set <= BE_VERBS:
        return 'Be verb (is/are/was)'

    # Word order (same words, different order)
    if sorted(rem) == sorted(add) and rem != add:
        return 'Word order'

    # Redundancy removal
    if not rem and add:
        return 'Missing word (insertion)'
    if rem and not add:
        return 'Redundant word (deletion)'

    # Capitalization
    if [w.lower() for w in removed_words] == [w.lower() for w in added_words]:
        return 'Capitalization'

    # Conjunction / transition
    if (rem_set | add_set) & CONJUNCTIONS and len(rem) <= 2:
        return 'Conjunction / connector'

    # Plural / singular
    if len(rem) == 1 and len(add) == 1:
        r, a = rem[0], add[0]
        if (r+'s' == a) or (a+'s' == r) or (r[:-1] == a) or (a[:-1] == r):
            return 'Singular / plural'

    # Default
    return 'Word choice / phrasing'


def analyze_patterns(df_in, max_rows=500):
    """Run diff on all pairs and return change records."""
    changes = []
    sample = df_in.head(max_rows)
    for _, row in sample.iterrows():
        orig = row['draft'][:600].split()
        corr = row['corrected'][:600].split()
        if not orig or not corr:
            continue
        matcher = difflib.SequenceMatcher(None, orig, corr, autojunk=False)
        for op, i1, i2, j1, j2 in matcher.get_opcodes():
            if op == 'equal':
                continue
            rem = orig[i1:i2]
            add = corr[j1:j2]
            cat = categorize_change(rem, add)
            changes.append({
                'date':     row['date'],
                'category': cat,
                'removed':  ' '.join(rem),
                'added':    ' '.join(add),
                'context':  ' '.join(orig[max(0,i1-3):i2+3]),
            })
    return pd.DataFrame(changes)

print("Analyzing patterns (may take ~30s)...")
changes_df = analyze_patterns(extracted)
print(f"Total changes analyzed: {len(changes_df)}")
changes_df.head()

In [ ]:
# --- Plot: error category frequency ---
cat_counts = changes_df['category'].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(cat_counts.index[::-1], cat_counts.values[::-1], color='steelblue')
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_xlabel('Number of corrections')
ax.set_title('Your Most Frequent English Correction Types', fontsize=13, pad=12)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'error_categories.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nTop categories:")
print(cat_counts.to_string())

In [ ]:
# --- Plot: error trend over time ---
changes_df['year_month'] = pd.to_datetime(changes_df['date'], errors='coerce').dt.to_period('M')
trend = (
    changes_df.dropna(subset=['year_month'])
    .groupby(['year_month','category'])
    .size()
    .unstack(fill_value=0)
)

top_cats = cat_counts.head(5).index
trend_top = trend[top_cats] if set(top_cats) <= set(trend.columns) else trend

fig, ax = plt.subplots(figsize=(12, 5))
trend_top.plot(ax=ax, marker='o', markersize=3, linewidth=1.5)
ax.set_xlabel('Month')
ax.set_ylabel('Correction count')
ax.set_title('Error Trends Over Time (Top 5 Categories)', fontsize=13)
ax.legend(loc='upper left', fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'error_trend.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Top specific corrections per category ---
print("=== Most frequent specific corrections ===\n")
for cat in cat_counts.head(6).index:
    sub = changes_df[changes_df['category'] == cat].copy()
    sub['pair'] = sub['removed'].str[:30] + ' → ' + sub['added'].str[:30]
    top = sub['pair'].value_counts().head(8)
    print(f"\n[{cat}]")
    for pair, cnt in top.items():
        print(f"  {cnt:3d}x  {pair}")

## 4 — Anki Flashcard Export

Exports two CSV files:
- `anki_corrections.csv` — Before/After pairs (one per correction)
- `anki_patterns.csv` — Error category cards with example sentences

**How to import into Anki:**
1. Open Anki → File → Import
2. Select the CSV file
3. Set field separator to comma
4. Map: Field 1 = Front, Field 2 = Back, Field 3 = Tags

In [ ]:
# --- Anki Card 1: Before/After pairs (from clean diffs) ---

def make_anki_pair_cards(changes_df, max_cards=500):
    """Create cards: Front=original phrase, Back=corrected phrase + category."""
    cards = []
    # Filter to meaningful changes (not too short, not too long)
    filtered = changes_df[
        (changes_df['removed'].str.len() > 2) &
        (changes_df['added'].str.len() > 2) &
        (changes_df['removed'] != '') &
        (changes_df['added'] != '')
    ].copy()

    # Deduplicate same corrections
    filtered['key'] = filtered['removed'].str.lower() + '|||' + filtered['added'].str.lower()
    seen = set()
    for _, row in filtered.iterrows():
        if row['key'] in seen:
            continue
        seen.add(row['key'])
        front = f"❌ {row['removed']}"
        back  = f"✅ {row['added']}<br><br><small>Category: {row['category']}<br>Context: ...{row['context']}...</small>"
        tag   = row['category'].lower().replace(' ','_').replace('/','_')
        cards.append([front, back, tag])
        if len(cards) >= max_cards:
            break
    return cards

pair_cards = make_anki_pair_cards(changes_df)
pair_path = os.path.join(OUT_DIR, 'anki_corrections.csv')
with open(pair_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['Front', 'Back', 'Tags'])
    w.writerows(pair_cards)
print(f"Saved {len(pair_cards)} pair cards → {pair_path}")

In [ ]:
# --- Anki Card 2: Pattern summary cards (one per category) ---

def make_pattern_cards(changes_df):
    """Create one card per error category with top examples."""
    cards = []
    for cat, grp in changes_df.groupby('category'):
        examples = (
            grp[grp['removed'].str.len() > 1]
            .assign(pair=grp['removed'].str[:40] + ' → ' + grp['added'].str[:40])
            ['pair'].value_counts().head(5)
        )
        ex_html = '<br>'.join([f'• {p} ({c}x)' for p, c in examples.items()])
        front = f"Pattern: <b>{cat}</b><br><small>({len(grp)} corrections total)</small>"
        back  = f"Common corrections:<br><br>{ex_html}"
        tag   = 'pattern_card ' + cat.lower().replace(' ','_').replace('/','_')
        cards.append([front, back, tag])
    return cards

pattern_cards = make_pattern_cards(changes_df)
pat_path = os.path.join(OUT_DIR, 'anki_patterns.csv')
with open(pat_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['Front', 'Back', 'Tags'])
    w.writerows(pattern_cards)
print(f"Saved {len(pattern_cards)} pattern cards → {pat_path}")
print("\n✅ Done! Import the CSV files into Anki:")
print(f"   1. {pair_path}")
print(f"   2. {pat_path}")

## 5 — Deep Dive: Spot-check by Category

Browse specific corrections for any error type.

In [ ]:
# Change CATEGORY to explore a specific type
CATEGORY = 'Article (a/an/the)'   # or 'Preposition', 'Word choice / phrasing', etc.

sub = changes_df[changes_df['category'] == CATEGORY]
print(f"Category: {CATEGORY} — {len(sub)} total corrections\n")

# Show top corrections with context
top = sub.assign(
    pair=sub['removed'].str[:40] + ' → ' + sub['added'].str[:40]
).groupby('pair').agg(
    count=('pair','size'),
    example=('context','first')
).sort_values('count', ascending=False).head(20)

for pair_str, row in top.iterrows():
    print(f"  {row['count']:3d}x  {pair_str}")
    print(f"         Context: ...{row['example'][:80]}...\n")